# 🎧 Sintonia — 1. Montar o catálogo

Este notebook monta o **catálogo**: o conjunto de músicas que o app pode recomendar.
Para cada música ele baixa o áudio, separa a voz com IA, mede melodia, voz, ritmo, harmonia e som, guarda os números num banco SQL **e apaga o áudio**.

**Como usar:** clique em cada bloco de código e aperte ▶️ (ou `Ctrl+Enter`), de cima para baixo.

> ⚡ **Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**. Com GPU a separação da voz fica umas 20× mais rápida.

## Passo 1 — Pegar o projeto

Escolha **uma** das opções:
- **A)** se você já subiu o projeto no GitHub, coloque o endereço em `REPO`.
- **B)** senão, deixe `REPO = ""` e envie o arquivo `sintonia.zip` quando aparecer o botão.

In [ ]:
import os

REPO = ""  # ex.: "https://github.com/SEU_USUARIO/sintonia.git"

os.chdir("/content")
if os.path.exists("/content/sintonia") and REPO:
    !git -C /content/sintonia pull -q   # já baixado antes: só atualiza
elif not os.path.exists("/content/sintonia"):
    if REPO:
        !git clone -q {REPO} /content/sintonia
    else:
        from google.colab import files
        enviado = files.upload()  # escolha o sintonia.zip
        nome_zip = next(iter(enviado))
        !unzip -q -o "{nome_zip}" -d /content
os.chdir("/content/sintonia")
print("Pasta do projeto:", os.getcwd())

## Passo 2 — Instalar as bibliotecas

- **demucs**: a IA que separa voz, bateria, baixo e o resto
- **librosa**: análise de áudio (notas, ritmo, acordes)
- **openai-whisper**: descobre o idioma cantado em cada música
- **umap-learn**: desenha o "mapa musical" no final

Leva uns 2 minutos.

In [ ]:
!pip install -q demucs librosa soundfile tqdm umap-learn openai-whisper

import torch
print("GPU disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ Sem GPU: vai funcionar, mas bem mais devagar. Veja a dica lá no topo.")

## Passo 3 — Salvar o progresso no Google Drive

O Colab apaga tudo quando a sessão acaba. Guardando o banco no seu Drive, se cair no meio você roda de novo e ele **continua de onde parou**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/sintonia"
os.makedirs(PASTA, exist_ok=True)
DB = f"{PASTA}/catalogo.db"
print("Banco do catálogo:", DB)

## Passo 4 — Chave do Jamendo

O Deezer não pede cadastro. O Jamendo pede um `client_id` **gratuito**:
1. Entre em https://devportal.jamendo.com e crie uma conta
2. Crie um app (pode chamar de "sintonia") e copie o **Client ID**
3. No Colab, clique na 🔑 (Secrets) na barra da esquerda, adicione `JAMENDO_CLIENT_ID` com o valor e ative o acesso para este notebook

Sem a chave, pule a parte do Jamendo.

In [ ]:
try:
    from google.colab import userdata
    os.environ["JAMENDO_CLIENT_ID"] = userdata.get("JAMENDO_CLIENT_ID")
    print("Chave do Jamendo carregada ✅")
except Exception:
    print("Sem chave do Jamendo — tudo bem, dá para usar só o Deezer.")

## Passo 5 — Testar com uma música

Antes de analisar centenas, vamos ver o que acontece com **uma**. Troque o nome se quiser.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
from sintonia import fontes
from sintonia.analise import analisar_faixa

resultados = fontes.buscar("deezer", "Evidências Chitãozinho e Xororó", limite=3)
for r in resultados:
    print("-", r.nome)

musica = resultados[0]
feats, info, partes, metodo = analisar_faixa(musica, metodo="demucs", manter_partes=True)
print(f"\nSeparação: {metodo} | tom estimado: {info['tom']} | notas cantadas: {info['n_notas']}")

### Ouvindo as partes separadas
É exatamente isso que a pessoa ouve no **teste de escuta** do app.

In [ ]:
from IPython.display import Audio, display
from sintonia.audio import SR

for parte in ["voz", "instrumental", "batida"]:
    print(parte)
    display(Audio(partes[parte], rate=SR))

### O desenho da melodia
Cada traço é uma nota cantada. O app compara os **intervalos** entre as notas (quanto sobe ou desce), então a mesma melodia em outro tom continua sendo reconhecida.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3))
for n in info["notas"]:
    ax.hlines(n["altura"], n["inicio"], n["fim"], linewidth=4)
ax.set_xlabel("tempo (s)"); ax.set_ylabel("altura (nota MIDI)")
ax.set_title(f"Melodia — {musica.nome}")
plt.show()

### Os números
Algumas das características que viram a "impressão digital" da música:

In [ ]:
from sintonia.descricoes import ROTULOS

for aspecto, valores in feats.items():
    print(f"\n== {aspecto.upper()}")
    for nome, valor in valores.items():
        if nome in ROTULOS:
            print(f"  {ROTULOS[nome]:<35} {valor:8.3f}")

## Passo 6 — Catálogo do Deezer

Busca playlists de vários estilos (MPB, sertanejo, rock, indie, soul...) e analisa as músicas.

- `600` músicas levam mais ou menos **1 hora** com GPU.
- Pode interromper (⏹️) e rodar de novo: ele pula as que já fez.
- Para mudar os estilos, passe `termos=["mpb", "indie folk", ...]`.

In [ ]:
from sintonia.catalogo import construir

construir("deezer", maximo=600, caminho_db=DB, metodo="demucs")

## Passo 7 — Catálogo do Jamendo (músicas completas)

Aqui a música é analisada **inteira**, então cada uma demora mais (uns 30 s). Comece com 150 e aumente depois.

In [ ]:
if os.environ.get("JAMENDO_CLIENT_ID"):
    construir("jamendo", maximo=150, caminho_db=DB, metodo="demucs")
else:
    print("Pulei o Jamendo (sem chave).")

## Passo 7b — Descobrir o idioma das músicas

O filtro de idioma do app (Português, Inglês, Espanhol, Japonês, Coreano...) precisa saber em que língua cada música é cantada. Quem descobre é o **Whisper**, um modelo de reconhecimento de fala, ouvindo a voz isolada de cada música.

As músicas analisadas a partir de agora já ganham o idioma automaticamente. Este bloco completa as que foram analisadas **antes** dessa novidade. Como o áudio não é guardado, ele baixa cada prévia de novo, e leva uns 3 a 4 segundos por música (umas 600 em cerca de 40 minutos). Pode parar e rodar de novo: ele pula as que já têm idioma.

In [ ]:
from sintonia.idioma import completar_catalogo

completar_catalogo(DB)

In [ ]:
# quantas músicas de cada idioma ficaram no catálogo
import sqlite3, pandas as pd
from sintonia.recomendador import Recomendador
Recomendador.do_banco(DB).contagem_idiomas().rename("músicas").to_frame()

## Passo 8 — Uma olhada com SQL

O catálogo é um banco **SQLite**. SQL é a linguagem para fazer perguntas a ele. Mais exemplos em `sql/consultas.sql` e no notebook 02.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(DB)
pd.read_sql("""
    SELECT fonte,
           COUNT(*)     AS musicas,
           SUM(tem_voz) AS com_voz
    FROM musicas
    GROUP BY fonte
""", con)

In [ ]:
# por que algumas falharam? (sem prévia, erro de download...)
pd.read_sql("SELECT substr(motivo, 1, 60) AS motivo, COUNT(*) AS qtd FROM falhas GROUP BY 1 ORDER BY 2 DESC LIMIT 10", con)

## Passo 9 — Levar o catálogo para o app e para o Power BI

Copia o banco para `dados/catalogo.db` (onde o app procura), gera os CSVs do Power BI e baixa tudo.

In [ ]:
import shutil
from sintonia.exportar import exportar

con.close()
os.makedirs("dados", exist_ok=True)  # a pasta pode não ter ido para o GitHub (estava vazia)
shutil.copy(DB, "dados/catalogo.db")
for arquivo in exportar("dados/catalogo.db"):
    print("gerado:", arquivo)

!cd /content/sintonia && zip -q -r /content/sintonia_resultado.zip dados/catalogo.db powerbi/dados
from google.colab import files
files.download("/content/sintonia_resultado.zip")

## Pronto! 🎉

No zip baixado estão:
- `dados/catalogo.db` → coloque na pasta `dados/` do projeto e suba para o GitHub (é só números, sem áudio)
- `powerbi/dados/*.csv` → abra no Power BI (veja `powerbi/README.md`)

Próximo: o notebook **02_explorar_catalogo** para explorar os dados com SQL e gráficos.